# Dialogue and Chatbots with Decoder-only Models (STUDENT version)

## 1. Title & introduction

In this notebook we build a **mini-chatbot** based on a **decoder-only Transformer model** (GPT / DialoGPT style) using the Hugging Face 🤗 `transformers` library.  
We focus on **inference**, not on heavy training or fine-tuning.

### Learning goals

By the end of this notebook, you should be able to:

- Load a **decoder-only model** from Hugging Face (we will use `microsoft/DialoGPT-small`).
- Implement a function that **generates replies** to user inputs.
- Build simple **multi-turn dialogue** using a text-based conversation history.
- Explore **decoding parameters** (greedy decoding, sampling, temperature, top-k, top-p).
- Use simple **prompting / system messages** to control the style or persona of the chatbot.
- Run a small **qualitative evaluation** of the generated dialogues.


## 2. (Optional) Library installation / upgrade

If you are running this notebook in **Google Colab** or in an environment where `transformers` is not up to date, you may want to install or upgrade it.

> 💡 If the import in the next section fails, come back here, **uncomment** the line below, run the cell, and then restart the kernel.


In [ ]:
# (Optional) Install or upgrade Hugging Face Transformers
# TODO (optional): Uncomment the following line if you are running in Google Colab
# !pip install -q "transformers>=4.40.0"


## 3. Imports and basic setup

In this section we:

- Import the necessary classes from **Hugging Face Transformers**:
  - `AutoTokenizer` and `AutoModelForCausalLM`.
- Import **PyTorch** (`torch`) and optionally **NumPy** (`numpy`) for numerical utilities.
- Select the computation **device**:
  - Use `"cuda"` if a GPU is available.
  - Fall back to `"cpu"` otherwise.
- (Optional) Set random seeds for reproducibility.


In [1]:
# Imports and basic setup

# TODO: import the classes and libraries we need.
# Hint: from transformers import AutoTokenizer, AutoModelForCausalLM
# Hint: import torch and (optionally) numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np

# TODO: set the device. Use "cuda" if a GPU is available, otherwise "cpu".
# Hint: device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"

# TODO (optional): set random seeds for reproducibility.
# Hint:
#   torch.manual_seed(42)
#   np.random.seed(42)
#   if torch.cuda.is_available():
#       torch.cuda.manual_seed_all(42)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# TODO: print the device to verify the setup.
print(f"Using device: {device}")

Using device: cuda


## 4. Loading the decoder-only model and tokenizer

We will use the pretrained model **`microsoft/DialoGPT-small`**, which is:

- A small **decoder-only Transformer** model (GPT-like).
- Trained on **English** conversational data.
- Designed specifically for **dialogue / chat** scenarios.

We load:

- The **tokenizer**, which converts between text and token IDs.
- The **causal language model**, which will generate replies.


In [2]:
# Loading the decoder-only model and tokenizer

# TODO: choose the model name.
model_name = "microsoft/DialoGPT-small"

# TODO: load the tokenizer.
# tokenizer = AutoTokenizer.from_pretrained(...)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# TODO: load the model and move it to the selected device.
# model = AutoModelForCausalLM.from_pretrained(...).to(device)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# TODO: put the model in evaluation mode.
# model.eval()
model.eval()

# TODO: print a short confirmation message with the model name and device.
print(f"Loaded model '{model_name}' on device '{device}'")

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded model 'microsoft/DialoGPT-small' on device 'cuda'


## 5. Single-turn generation

As a first test, we will perform **single-turn generation**:

- Start from a simple English prompt, such as:  
  `"Hello, how are you today?"`
- Tokenize the prompt and send it through the model via `model.generate(...)`.
- Use **greedy decoding** first (set `do_sample=False`).
- Optionally, try **sampling** (set `do_sample=True` and play with `temperature`, `top_k`, `top_p`).

We will then print:

- The original prompt.
- The full model output, including the continuation it generated.


In [3]:
# Single-turn generation with a fixed prompt

# TODO: define an English prompt, e.g.:
# prompt = "Hello, how are you today?"
prompt = "Hello, how are you today?"

# TODO: tokenize the prompt.
# Hint: use tokenizer(prompt, return_tensors="pt")
inputs = tokenizer(prompt, return_tensors="pt")

# TODO: move tensors to the correct device.
# Hint: inputs = {k: v.to(device) for k, v in inputs.items()}
inputs = {k: v.to(device) for k, v in inputs.items()}

# TODO: generate a continuation using greedy decoding.
# Hint: use model.generate with max_new_tokens=50 and do_sample=False.
generated_tokens = model.generate(**inputs, max_new_tokens=50, do_sample=False)

# TODO: decode the generated tokens to text.
# Hint: use tokenizer.decode(..., skip_special_tokens=True).
completion = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

# TODO: print the original prompt and the model's completion.
print(f"Prompt: {prompt}")
print(f"Completion: {completion}")

# Optional: try a second generation using sampling (do_sample=True, temperature, top_p, top_k).

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: Hello, how are you today?
Completion: Hello, how are you today?


## 6. Representing the conversation history

To handle **multi-turn dialogue**, we need a way to represent the **conversation history**.

A simple and flexible approach is to store messages as a list of strings, for example:

- `"System: You are a helpful assistant."`
- `"User: Hello!"`
- `"Bot: Hi! How can I help you?"`

We will create a helper function:

```python
build_prompt(history, max_turns=4)
```

that:

1. Optionally truncates the history to the last `max_turns` messages.
2. Joins the selected messages with newline characters (`"\n"`).
3. Returns the resulting string, which we will feed to the model.


In [4]:
# Representing the conversation history

# We represent each message as a string like "User: ..." or "Bot: ...".
# TODO: implement the function build_prompt(history, max_turns=4).
# - history: list of strings.
# - max_turns: maximum number of most recent messages to include (if None, include all).

def build_prompt(history, max_turns=4):
    '''
    Build a text prompt from the conversation history.

    Args:
        history (list of str): messages like "User: ...", "Bot: ...", "System: ...".
        max_turns (int or None): number of last messages to keep. If None, keep all.

    Returns:
        str: the joined prompt.
    '''
    # TODO: if max_turns is not None, keep only the last max_turns messages.
    # Hint: use list slicing: history[-max_turns:]
    if max_turns is not None:
        history = history[-max_turns:]

    # TODO: join the selected messages with newline characters ("\n").
    # Hint: use "\n".join(...)
    prompt = "\n".join(history)

    # TODO: return the resulting string.
    return prompt

# TODO: create a small example history and print the prompt.
# Example:
# history_example = [
#     "System: You are a helpful assistant.",
#     "User: Hello!",
#     "Bot: Hi! How can I help you?",
#     "User: Can you tell me a joke?"
# ]
# prompt_example = build_prompt(history_example, max_turns=4)
# print(prompt_example)

history_example = [
    "System: You are a helpful assistant.",
    "User: Hello!",
    "Bot: Hi! How can I help you?",
    "User: Can you tell me a joke?"
]

prompt_example = build_prompt(history_example, max_turns=4)
print(prompt_example)

System: You are a helpful assistant.
User: Hello!
Bot: Hi! How can I help you?
User: Can you tell me a joke?


## 7. One-turn chat function (`chat_once`) using only newly generated tokens

We now create a function that performs **one turn** of conversation, updating the history and returning the bot's reply.

The key design choice is that we want to decode **only the new tokens** generated by the model, **not** the entire input plus output.

We will implement:

```python
chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs)
```

Steps:

1. Append `"User: {user_utterance}"` to `history`.
2. Build the full prompt using `build_prompt(history, max_turns=None)` (keep all messages).
3. Tokenize the prompt:
   - `inputs = tokenizer(prompt, return_tensors="pt")`
4. Extract `input_ids = inputs["input_ids"]` and move everything to `device`.
5. Call `model.generate(...)` with:
   - `input_ids=input_ids`
   - `attention_mask=inputs.get("attention_mask", None)`
   - `max_new_tokens`, `do_sample`, `temperature`, `top_k`, `top_p`, etc.
6. Extract only the **newly generated tokens**:
   - `generated_ids = output_ids[:, input_ids.shape[1]:]`
7. Decode these new tokens:
   - `bot_reply = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()`
8. If `bot_reply` is empty, replace it with a fallback string such as:
   - `"(I did not manage to generate a reply...)"`
9. Append `"Bot: {bot_reply}"` to `history`.
10. Return the updated `(history, bot_reply)`.


In [5]:
# One-turn chat function using only newly generated tokens

# TODO: implement chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs)
# Follow the steps described in the markdown above.

def chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs):
    '''
    Perform one turn of dialogue.

    Args:
        history (list of str): conversation so far.
        user_utterance (str): new user message.
        max_new_tokens (int): maximum number of new tokens to generate.
        **decoding_kwargs: additional generation arguments such as
            do_sample, temperature, top_k, top_p, etc.

    Returns:
        history (list of str): updated history including bot reply.
        bot_reply (str): the decoded reply from the model.
    '''
    # TODO: 1) append the user message to history.
    history.append(f"User: {user_utterance}")
    # TODO: 2) build the full prompt (e.g., with max_turns=None to keep all messages).
    prompt = build_prompt(history, max_turns=None)
    # TODO: 3) tokenize and move tensors to device.
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    # TODO: 4) call model.generate with input_ids and attention_mask.
    generated_tokens = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        **decoding_kwargs
    )
    # TODO: 5) slice out only the new tokens using input_ids.shape[1].
    new_tokens = generated_tokens[0][inputs["input_ids"].shape[1]:]
    # TODO: 6) decode the new tokens.
    bot_reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    # TODO: 7) handle empty replies with a fallback string.
    if not bot_reply.strip():
        bot_reply = "<no reply>"
    # TODO: 8) append the bot reply to history.
    history.append(f"Bot: {bot_reply}")
    # TODO: 9) return history and bot_reply.
    return history, bot_reply


# TODO: quick test of chat_once.
# Example:
# history = []
# history, reply = chat_once(history, "Hi! How are you?")
# print("Bot:", reply)
# print("History:", history)

history = []
history, reply = chat_once(history, "Hi! How are you?")
print("Bot:", reply)
print("History:", history)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Bot: !
History: ['User: Hi! How are you?', 'Bot: !']


## 8. Experiments with decoding parameters

Now we will explore how **decoding parameters** affect the responses.

Choose a fixed user input, for example:

```text
"Can you explain what a Transformer model is in simple terms?"
```

Then define several decoding configurations, for instance:

- **Greedy**:
  - `do_sample=False`
- **Sampling (moderate creativity)**:
  - `do_sample=True, temperature=0.7, top_p=0.9`
- **Sampling (more creative)**:
  - `do_sample=True, temperature=1.2, top_k=50`

For each configuration:

1. Start from an **empty history** (or include a system prompt if you prefer).
2. Call `chat_once(...)` with the desired decoding parameters.
3. Compare the replies in terms of coherence, relevance, diversity, and length.


In [7]:
# Experiments with decoding parameters

# TODO: choose a user input for comparison, e.g.:
# user_input = "Can you explain what a Transformer model is in simple terms?"
user_input = "Can you explain what a Transformer model is in simple terms?"

# TODO: define a list of decoding configurations to compare.
# Example structure:
# configs = [
#     {"name": "greedy", "do_sample": False},
#     {"name": "sampling_temp0.7_top_p0.9", "do_sample": True, "temperature": 0.7, "top_p": 0.9},
#     {"name": "sampling_temp1.2_top_k50", "do_sample": True, "temperature": 1.2, "top_k": 50},
# ]

configs = [
    {"name": "greedy", "do_sample": False},
    {"name": "sampling_temp0.7_top_p0.9", "do_sample": True, "temperature": 0.7, "top_p": 0.9},
    {"name": "sampling_temp1.2_top_k50", "do_sample": True, "temperature": 1.2, "top_k": 50},
]

# TODO: for each configuration:
#   - start from an empty history (or include a system prompt if you want),
#   - call chat_once with the configuration parameters,
#   - print the configuration name and the bot's reply.

# Hint:
# for cfg in configs:
#     print("=" * 80)
#     print("Config:", cfg["name"])
#     ...

for cfg in configs:
    print("=" * 80)
    print("Config:", cfg["name"])
    history = []
    # Extract name separately and pass only the decoding parameters
    cfg_params = {k: v for k, v in cfg.items() if k != "name"}
    history, reply = chat_once(history, user_input, **cfg_params)
    print("Bot:", reply)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Config: greedy
Bot: <no reply>
Config: sampling_temp0.7_top_p0.9
Bot: <no reply>
Config: sampling_temp1.2_top_k50
Bot:  The question that popped up in my head.


## 9. Controlling chatbot style via prompting

We can influence the **style** or **persona** of the chatbot using a special **system message** in the history.

For example:

- Formal assistant:

  ```text
  "System: You are a polite and formal assistant. You answer concisely and respectfully."
  ```

- Friendly / funny assistant:

  ```text
  "System: You are a friendly and slightly funny assistant. You sometimes make small jokes."
  ```

We will:

1. Create separate histories for each style.
2. Run the same set of user prompts through both histories using `chat_once`.
3. Compare the differences in **tone** and **style** of the replies.


In [8]:
# Controlling chatbot style via prompting

# TODO: define two different system prompts that specify different styles.
# Example:
# system_formal = "System: You are a polite and formal assistant. You answer concisely and respectfully."
# system_funny  = "System: You are a friendly and slightly funny assistant. You sometimes make small jokes."
system_formal = "System: You are a polite and formal assistant. You answer concisely and respectfully."
system_funny  = "System: You are a friendly and slightly funny assistant. You sometimes make small jokes."

# TODO: initialize two separate histories using these system prompts.
# history_formal = [system_formal]
# history_funny  = [system_funny]
history_formal = [system_formal]
history_funny  = [system_funny]

# TODO: define a small list of user messages you want to test, e.g.:
# user_messages = [
#     "Hello! Could you introduce yourself?",
#     "What is a Transformer model?",
#     "Do you have any study tips?"
# ]
user_messages = [
    "Hello! Could you introduce yourself?",
    "What is a Transformer model?",
    "Do you have any study tips?"
]

# TODO: for each user message, call chat_once with the corresponding history and print the result.
# Hint: compare how the style changes between the formal and funny histories.
for msg in user_messages:
    print("=" * 80)
    print("User message:", msg)

    # Formal style
    history_formal, reply_formal = chat_once(history_formal, msg)
    print("Formal Bot:", reply_formal)

    # Funny style
    history_funny, reply_funny = chat_once(history_funny, msg)
    print("Funny Bot:", reply_funny)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


User message: Hello! Could you introduce yourself?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Formal Bot:  User : I'm a software engineer. User : I'm a software engineer. User : I'm a software engineer. User : I'm a software engineer. User : I'm a software engineer.
Funny Bot: <no reply>
User message: What is a Transformer model?
Formal Bot:  User : I'm a software engineer. User : I'm a software engineer. User : I'm a software engineer.
Funny Bot: <no reply>
User message: Do you have any study tips?
Formal Bot: <no reply>
Funny Bot: <no reply>


## 10. Mini qualitative evaluation

In this section we will perform a **lightweight qualitative evaluation** of the chatbot responses.

You should:

1. Pick at least **5 user–bot exchanges** from your experiments above (for example, a few from different decoding settings and different personas).
2. Evaluate each interaction on several dimensions, using a 1–5 scale (1 = very poor, 5 = excellent).

Example dimensions:

- **Coherence**: Is the reply internally consistent and logically connected?
- **Relevance**: Does it address the user question or topic?
- **Fluency**: Is the text grammatically correct and natural-sounding?
- **Tone / Politeness**: Is the tone appropriate and polite?

You can use a table like this (feel free to copy and modify it in your report or notes):

| # | Interaction summary                                   | Coherence (1–5) | Relevance (1–5) | Fluency (1–5) | Tone / Politeness (1–5) |
|---|-------------------------------------------------------|-----------------|-----------------|---------------|-------------------------|
| 1 | User asks about Transformers; bot gives short answer. | 4               | 5               | 4             | 5                       |
| 2 |                                                       |                 |                 |               |                         |
| 3 |                                                       |                 |                 |               |                         |
| 4 |                                                       |                 |                 |               |                         |
| 5 |                                                       |                 |                 |               |                         |

After filling the table, write a short **comment (5–10 lines)** about:

- The **strengths** of the model in dialogue.
- The most common **weaknesses or errors** you observed.
- How decoding parameters and persona prompts influenced the behaviour.


## 11. Interactive chat loop (ChatGPT-style) – optional

As an optional extension, we can create a small **interactive chat loop** that behaves like a very simple version of ChatGPT.

We will implement:

```python
chat_loop(system_prompt="System: You are a helpful assistant.", ...)
```

The function will:

1. Initialize a `history = [system_prompt]`.
2. Print a welcome message, e.g. `"=== Mini chat (type 'exit' or 'quit' to stop) ==="`.
3. Enter a `while True` loop:
   - Read user input using `input("You: ")`.
   - If the input is `"exit"` or `"quit"` (case-insensitive), print a goodbye message and break.
   - Otherwise, call `chat_once(history, user_utt, ...)` and print `"Bot: ..."`.


> ⚠️ In some notebook environments (especially when running as a non-interactive job), `input()` may raise an `EOFError`. In the solved version, we will catch this error and stop the loop gracefully.


In [ ]:
# Optional interactive chat loop

# TODO: implement chat_loop(system_prompt="System: You are a helpful assistant.", max_new_tokens=80, **decoding_kwargs)
# Follow the steps described in the markdown above.

def chat_loop(system_prompt="System: You are a helpful assistant.", max_new_tokens=80, **decoding_kwargs):
    '''
    Simple interactive chat loop. Works best in a terminal or notebook cell with stdin.
    '''
    # TODO: initialize history with the system prompt.
    # TODO: print a welcome message explaining how to exit.
    # TODO: implement a while True loop with input("You: ").
    #   - break if the user types 'exit' or 'quit' (case-insensitive),
    #   - otherwise, call chat_once and print the bot reply.
    # Optional: catch EOFError to handle environments without interactive input.

    raise NotImplementedError("Implement chat_loop(...)")


# TODO (optional): Uncomment the line below to try the chat loop in an interactive environment.
# chat_loop()
